# Avito: кандидатогенерация для поиска услуг (Recall@50)

Задача: по тексту запроса и его признакам выбрать из корпуса `benchmark_items.parquet` до 50 объявлений так, чтобы в них попало как можно больше подходящих. Порядок внутри 50 не важен, считается только полнота.

Что в итоге получилось (подробности ниже по ноутбуку):

- ищем по всему корпусу, каждому объявлению считаем общий скор как взвешенную сумму нескольких сигналов;
- сигналы: BM25 по тексту и по заголовку, вероятность локации объявления при локации поиска (посчитана по train), расстояние, совпадение фильтров из `search_infm_params_text` с параметрами объявления, совпадение категории, подкатегория и расширение запроса по похожим запросам из train, эмбеддинги готовой модели и эмбеддинги двух моделей, дообученных на парах "запрос - объявление" из train (`rubert-tiny-turbo` и `rubert-mini-frida`);
- 150 лучших по этой сумме переранжирует градиентный бустинг, в ответ идут 50 лучших после него;
- веса и бустинг подобраны по Recall@50 на своей валидации (часть train) и проверены на второй отложенной выборке;
- на валидации модели дообучаются только на обучающей части train, а для финального ответа обе модели дообучаются заново на всем train (как и вся остальная статистика для ответа);
- валидация несколько раз переделывалась, чтобы быть похожей на benchmark: корпус с похожими объявлениями рядом с каждым запросом (раздел 6) и такой же состав запросов, как в benchmark (раздел 4). Проверка калибровки - два уже отправленных решения: на платформе 0.877 и 0.882, на отложенной выборке B они получают 0.880 и 0.892;
- итог на B: взвешенная сумма 0.887, с бустингом 0.902;
- в конце `answer.csv` проходит проверки из чек-листа.

Запуск: положить три parquet-файла в папку `dataset/` и выполнить ноутбук сверху вниз. На 8-ядерном CPU без GPU полный прогон с нуля занимает около 4 часов, из них примерно 2.5 часа - дообучение четырех моделей (две модели, каждая на обучающей части и на всем train). Эмбеддинги и дообученные модели сохраняются в `cache/`, поэтому повторный прогон быстрее.

## 0. Импорты и настройки

In [ ]:
import os
import re
import time
import hashlib
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import snowballstemmer
import torch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

SEED = 42
DATA_DIR = 'dataset'
CACHE_DIR = 'cache'
EMB_MODEL_NAME = 'sergeyzh/rubert-tiny-turbo'
FRIDA_MODEL_NAME = 'sergeyzh/rubert-mini-frida'
FRIDA_QUERY_PROMPT = 'search_query: '
FRIDA_DOC_PROMPT = 'search_document: '

os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)
start_time = time.time()

## 1. Загрузка данных и проверка типов

Сначала проверяю, что файлы на месте и колонки называются так, как в описании. Отдельно смотрю на ID: они должны остаться строками длиной 16 символов. `item_id` состоит из `0-9a-f`, а у `query_id` в данных встречаются и заглавные латинские буквы, поэтому для него проверяю только длину и тип.

In [ ]:
for name in ['train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet']:
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Не найден файл {path}')

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
queries = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_queries.parquet'))
items = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_items.parquet'))
print('train:', train.shape)
print('queries:', queries.shape)
print('items:', items.shape)

In [ ]:
search_cols = ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
item_cols = ['item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text', 'item_category_id',
             'item_microcat_id', 'item_price', 'item_rating', 'item_rating_reviews_count', 'item_location_id',
             'item_latitude', 'item_longitude', 'item_is_phone_hidden', 'item_is_message_forbidden']

missing = {
    'train': set(search_cols + item_cols) - set(train.columns),
    'queries': set(['query_id'] + search_cols) - set(queries.columns),
    'items': set(item_cols) - set(items.columns),
}
print(missing)
assert all(len(m) == 0 for m in missing.values())

In [ ]:
def check_ids(ids, name, hex_only):
    assert ids.map(type).eq(str).all(), name
    assert ids.str.len().eq(16).all(), name
    if hex_only:
        assert ids.str.fullmatch('[0-9a-f]{16}').all(), name
    print(f'{name}: dtype={ids.dtype}, пример={ids.iloc[0]}, пропусков={ids.isna().sum()}')


check_ids(queries['query_id'], 'query_id', hex_only=False)
check_ids(items['item_id'], 'item_id (корпус)', hex_only=True)
check_ids(train['item_id'], 'item_id (train)', hex_only=True)

Текстовые поля иногда пустые (например, 35 описаний в корпусе). Чтобы дальше ничего не падало на `None`, сразу заменяю пропуски в текстах на пустую строку.

In [ ]:
for df in [train, queries]:
    for col in ['search_query', 'search_infm_params_text']:
        df[col] = df[col].fillna('')
for df in [train, items]:
    for col in ['item_title_raw', 'item_description_raw', 'item_infm_params_text']:
        df[col] = df[col].fillna('')

print(queries.isna().sum().sum(), items[['item_title_raw', 'item_description_raw', 'item_infm_params_text']].isna().sum().sum())

## 2. EDA

In [ ]:
print('строк в train:', len(train))
print('уникальных текстов запросов в train:', train['search_query'].nunique())
print('уникальных комбинаций всех search_* полей в train:', train[search_cols].drop_duplicates().shape[0])
print('уникальных объявлений в train:', train['item_id'].nunique())
print()
print('запросов в benchmark:', len(queries), '| уникальных query_id:', queries['query_id'].nunique(), '| уникальных текстов:', queries['search_query'].nunique())
print('объявлений в корпусе:', len(items), '| уникальных item_id:', items['item_id'].nunique())
print('объявлений корпуса, которые встречаются в train:', items['item_id'].isin(set(train['item_id'])).sum())
seen_share = queries['search_query'].isin(set(train['search_query'])).mean()
print('доля запросов benchmark, текст которых был в train:', round(seen_share, 3))

In [ ]:
lengths = pd.DataFrame({
    'слов в запросе': queries['search_query'].str.split().str.len().describe(),
    'символов в заголовке': items['item_title_raw'].str.len().describe(),
    'символов в описании': items['item_description_raw'].str.len().describe(),
    'символов в параметрах': items['item_infm_params_text'].str.len().describe(),
})
lengths.round(1)

In [ ]:
print('пустой search_infm_params_text в train:', round((train['search_infm_params_text'] == '').mean(), 3))
print('пустой search_infm_params_text в benchmark:', round((queries['search_infm_params_text'] == '').mean(), 3))
print('пустых описаний в корпусе:', (items['item_description_raw'] == '').sum())
print('пустых заголовков в корпусе:', (items['item_title_raw'] == '').sum())
print('запросов из одного слова в benchmark:', (queries['search_query'].str.split().str.len() == 1).sum())

In [ ]:
print(pd.crosstab(train['search_category'], train['item_category_id']))
print()
print('search_category в benchmark:', queries['search_category'].value_counts().to_dict())
print('item_category_id в корпусе (топ-10):', items['item_category_id'].value_counts().head(10).to_dict())

In [ ]:
print('дублей строк (запрос + объявление) в train:', train.duplicated(search_cols + ['item_id']).sum())
print('объявлений в корпусе с таким же заголовком и описанием, как у другого:', items.duplicated(['item_title_raw', 'item_description_raw']).sum())

In [ ]:
same_loc = (train['search_location_id'] == train['item_location_id']).mean()
print('доля пар в train, где локация поиска совпадает с локацией объявления:', round(same_loc, 3))
print('локации поиска, у которых чаще всего выбирают объявление из другой локации:')
print(train.loc[train['search_location_id'] != train['item_location_id'], 'search_location_id'].value_counts().head(5))

In [ ]:
train['search_infm_params_text'].value_counts().head(12)

Что видно из EDA:

- Почти все запросы и объявления в train из категории 114 (услуги). В корпусе есть около 1.9 тыс. объявлений из других категорий. У 222 запросов benchmark (9%) `search_category = 0`, то есть поиск шел без категории, а в train таких запросов всего 34. Для них в разделе 8 отдельно проверяю, есть ли в корпусе подходящие объявления из других категорий.
- Запросы короткие, медиана 3 слова. Описания объявлений длинные, в параметрах много служебного текста (график работы и т.п.).
- Все тексты запросов в benchmark разные, и только около 37% из них встречались в train. Значит, простое запоминание пар "запрос - объявление" не спасет, нужен поиск по тексту.
- Объявления корпуса почти не пересекаются с train (около 18 тыс. из 189 тыс.).
- Очень сильный сигнал - локация: в 83% пар локация поиска совпадает с локацией объявления. Остальное в основном "регион -> город" (например, поиск по области, а объявление в областном центре). Такие переходы можно посчитать по train.
- `search_infm_params_text` - это фильтры поиска ("Вид услуги ...", "Тип услуги ..."). У 37% запросов benchmark они заполнены, и их можно сверять с `item_infm_params_text`. В train фильтры есть у 65% комбинаций, то есть запросы benchmark по составу отличаются от train. Это учитываю при сборке валидации.
- Есть дубли строк в train и объявления с одинаковым текстом в корпусе. Дубли строк при группировке по запросу схлопываются, а дубли объявлений проверю в конце на ответе.

## 3. Подготовка текста

Токенизация простая: нижний регистр, `ё -> е`, слова из букв и цифр, затем стемминг Snowball для русского (чтобы "дровах" и "дрова" совпали). Эмодзи и прочие символы просто выкидываются, пустая строка дает пустой результат.

In [ ]:
stemmer = snowballstemmer.stemmer('russian')
token_re = re.compile(r'[a-zа-я0-9]+')
stem_cache = {}


def stem_text(text):
    words = token_re.findall(text.lower().replace('ё', 'е'))
    result = []
    for w in words:
        if w not in stem_cache:
            stem_cache[w] = stemmer.stemWord(w)
        result.append(stem_cache[w])
    return ' '.join(result)


examples = ['Баня на дровах', 'ремонт iPhone 11 \U0001F600', '', '   ', '№1!!! Ёлка-палка, 5-10 м²']
for example in examples:
    print(repr(stem_text(example)))

## 4. Валидация: сплит и метрика

У benchmark нет разметки, поэтому качество проверяю на части train.

- Делю по уникальным комбинациям всех `search_*` полей, а не по строкам: одна и та же комбинация не может оказаться и в обучающей, и в валидационной части. 40% комбинаций уходит в валидацию. Доля большая, потому что из валидационной части потом берутся "похожие" объявления для усложнения валидации (раздел 6), а для статистики по train хватает и оставшихся 60%.
- Все объявления, выбранные по одной комбинации, собираю в один валидационный запрос с несколькими релевантными `item_id`.
- В benchmark у каждого запроса свой текст, поэтому беру по одной комбинации на текст.
- Состав запросов подбираю таким же, как в benchmark, по трем признакам: насколько часто текст встречался в train, есть ли фильтры и из крупного ли города запрос. Это важно: в benchmark 63% запросов без фильтров (в train 35%), 22% из двух самых крупных городов, а знакомые тексты в основном редкие (медиана 3 строки в train). Если брать валидационные запросы случайно, знакомые тексты там оказываются популярными (медиана около 20 строк), а по популярным запросам статистика из train намного точнее, и оценка получается завышенной. Обучающая часть - это 60% train, поэтому для запросов benchmark число строк в train умножаю на 0.6, чтобы сравнивать одинаково.
- Делаю две непересекающиеся выборки по ~3000 запросов: A для подбора весов и обучения бустинга, B для честной проверки.
- Корпус для поиска на валидации: весь `benchmark_items`, релевантные объявления валидационных запросов и "похожие" объявления, которые подбираются в разделе 6. То есть нужные объявления ищутся среди ~270 тыс. других, как и в настоящей задаче.
- Вся статистика (переходы локаций, похожие запросы и т.д.) и дообучение моделей на валидации используют только обучающую часть.

In [ ]:
def recall_at_50(predictions, relevant):
    scores = []
    for pred, rel in zip(predictions, relevant):
        rel = set(rel)
        scores.append(len(set(pred[:50]) & rel) / len(rel))
    return float(np.mean(scores))


example_pred = [['a1'], ['b1', 'x1'], ['z1']]
example_rel = [['a1'], ['b1', 'b2'], ['c1']]
print('пример из задания:', recall_at_50(example_pred, example_rel))
assert abs(recall_at_50(example_pred, example_rel) - 0.5) < 1e-12
assert recall_at_50([[str(i) for i in range(100)]], [['0', '75']]) == 0.5
assert recall_at_50([['a', 'a', 'b']], [['a', 'b']]) == 1.0

In [ ]:
combos = train[search_cols].drop_duplicates().reset_index(drop=True)
rng = np.random.RandomState(SEED)
combos['is_val'] = rng.rand(len(combos)) < 0.4
train = train.merge(combos, on=search_cols, how='left')
assert train['is_val'].notna().all() and train['item_id'].dtype == items['item_id'].dtype

train_part = train[~train['is_val']].reset_index(drop=True)
val_part = train[train['is_val']].reset_index(drop=True)
print('обучающая часть:', len(train_part), 'строк | валидационная часть:', len(val_part), 'строк')

val_queries = val_part.groupby(search_cols)['item_id'].apply(lambda s: sorted(set(s))).reset_index(name='relevant')
val_queries = val_queries.sample(frac=1, random_state=SEED).drop_duplicates('search_query').reset_index(drop=True)
val_queries['seen'] = val_queries['search_query'].isin(set(train_part['search_query']))

big_cities = queries['search_location_id'].value_counts().index[:2].tolist()
part_text_counts = train_part['search_query'].value_counts()
train_text_counts = train['search_query'].value_counts()


def query_group(df, text_counts, scale):
    n = text_counts.reindex(df['search_query']).fillna(0).values * scale
    popularity = np.select([n == 0, n <= 2, n <= 6], ['новый текст', 'редкий', 'средний'], 'частый')
    filters = np.where(df['search_infm_params_text'] == '', 'без фильтров', 'с фильтрами')
    city = np.where(df['search_location_id'].isin(big_cities), 'крупный город', 'другой город')
    return pd.Series(popularity, index=df.index).astype(str) + ' / ' + filters + ' / ' + city


val_queries['group'] = query_group(val_queries, part_text_counts, 1.0)
bench_group_share = query_group(queries, train_text_counts, 0.6).value_counts(normalize=True)
parts_a, parts_b = [], []
for group, share in bench_group_share.items():
    candidates = val_queries[val_queries['group'] == group]
    n = int(round(share * 3000))
    assert len(candidates) >= 2 * n, group
    parts_a.append(candidates.iloc[:n])
    parts_b.append(candidates.iloc[n:2 * n])
val_a = pd.concat(parts_a).reset_index(drop=True)
val_b = pd.concat(parts_b).reset_index(drop=True)
print('A:', len(val_a), 'запросов, доля виденных', round(val_a['seen'].mean(), 3), ', релевантных на запрос', round(val_a['relevant'].str.len().mean(), 2))
print('B:', len(val_b), 'запросов, доля виденных', round(val_b['seen'].mean(), 3), ', релевантных на запрос', round(val_b['relevant'].str.len().mean(), 2))
assert not set(val_a['search_query']) & set(val_b['search_query'])
pd.DataFrame({'benchmark': bench_group_share, 'A': val_a['group'].value_counts(normalize=True),
              'B': val_b['group'].value_counts(normalize=True), 'всего в валидационной части': val_queries['group'].value_counts()}).round(3)

Собираю все объявления, которые понадобятся дальше: `benchmark_items`, релевантные объявления выборок A и B и объявления из валидационной части, которых нет в обучающей части. Из последних в разделе 6 выбираются "похожие" объявления для валидации.

In [ ]:
bench_ids = set(items['item_id'])
val_relevant_ids = set(i for lst in pd.concat([val_a, val_b])['relevant'] for i in lst)
val_items = val_part.drop_duplicates('item_id')[item_cols]
val_items = val_items[~val_items['item_id'].isin(bench_ids)]
cold_ids = set(val_items['item_id']) - set(train_part['item_id'])
extra_items = val_items[val_items['item_id'].isin(val_relevant_ids | cold_ids)]
all_corpus = pd.concat([items[item_cols], extra_items]).reset_index(drop=True)
n_bench_items = len(items)
assert all_corpus['item_id'].is_unique
assert (all_corpus['item_id'].iloc[:n_bench_items].values == items['item_id'].values).all()
print('всего объявлений:', len(all_corpus), '| из benchmark:', n_bench_items, '| релевантных не из benchmark:', len(val_relevant_ids - bench_ids), '| нет в обучающей части:', len(cold_ids))

Стемминг текстов корпуса. Первые 189 212 строк `all_corpus` - это ровно `benchmark_items`, поэтому для финального ответа потом просто беру эту часть.

In [ ]:
t = time.time()
all_corpus['title_st'] = [stem_text(x) for x in all_corpus['item_title_raw']]
all_corpus['full_st'] = [a + ' ' + stem_text(b) + ' ' + stem_text(c) for a, b, c in
                         zip(all_corpus['title_st'], all_corpus['item_description_raw'], all_corpus['item_infm_params_text'])]
title_st_map = {}
for item_id, title in zip(train['item_id'], train['item_title_raw']):
    if item_id not in title_st_map:
        title_st_map[item_id] = stem_text(title)
train['title_st'] = train['item_id'].map(title_st_map)
train_part['title_st'] = train_part['item_id'].map(title_st_map)
for df in [val_a, val_b, queries]:
    df['query_st'] = [stem_text(x) for x in df['search_query']]
val_ab = pd.concat([val_a.assign(part='A'), val_b.assign(part='B')]).reset_index(drop=True)
print('стемминг занял', round(time.time() - t), 'сек')

## 5. Эмбеддинги готовой модели

Беру открытую модель `sergeyzh/rubert-tiny-turbo` (маленький русский BERT, 312-мерные эмбеддинги). Она самая быстрая на CPU. Я замерял скорость на 8 ядрах (объявлений в секунду при длине 48 токенов): rubert-tiny-turbo 830, rubert-mini-frida 380, USER2-small 130, multilingual-e5-small 120, BERTA 36. Модели больше mini-frida считали бы корпус из 300 тыс. объявлений от 40 минут до нескольких часов, поэтому их не беру. Модель скачивается с Hugging Face один раз при первом запуске, дальше все считается локально без обращений к сети.

Текст объявления для эмбеддинга: заголовок + первые 300 символов описания. Результаты сохраняю в `cache/`, в имя файла добавляю хэш от входных текстов, чтобы случайно не взять устаревший кэш.

In [ ]:
base_model = SentenceTransformer(EMB_MODEL_NAME, device='cpu')


def encode(model, texts, name, max_len):
    key = hashlib.md5(('\n'.join(texts) + name).encode('utf-8')).hexdigest()[:12]
    path = os.path.join(CACHE_DIR, f'{name}_{key}.npy')
    if os.path.exists(path):
        return np.load(path)
    model.max_seq_length = max_len
    emb = model.encode(texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False, prompt='')
    emb = emb.astype(np.float32)
    np.save(path, emb)
    return emb


t = time.time()
item_texts = (all_corpus['item_title_raw'] + '. ' + all_corpus['item_description_raw'].str.slice(0, 300)).tolist()
item_emb = encode(base_model, item_texts, 'items', 128)
all_query_texts = sorted(set(train['search_query']) | set(val_ab['search_query']) | set(queries['search_query']))
query_emb = encode(base_model, all_query_texts, 'queries', 128)
query_emb_pos = pd.Series(np.arange(len(all_query_texts)), index=all_query_texts)
print(item_emb.shape, query_emb.shape, 'за', round(time.time() - t), 'сек')

## 6. Почему первая валидация была слишком легкой и как это исправлено

Первая версия этого ноутбука строила корпус для валидации просто как `benchmark_items` + релевантные объявления валидационных запросов. На такой валидации получалось Recall@50 около 0.907, а на платформе та же модель получила **0.877**. Разница большая, значит валидация была устроена не так, как benchmark.

Чтобы понять разницу, считаю для каждого запроса, сколько в корпусе объявлений из **той же локации**, которые **похожи на запрос по смыслу** (косинус эмбеддингов больше 0.7 и больше 0.8). Для запросов benchmark считаю по `benchmark_items`, для валидационных запросов - по тому корпусу, который был в первой версии.

In [ ]:
item_loc = all_corpus['item_location_id'].values


def close_items(qdf, positions):
    q = query_emb[query_emb_pos.loc[qdf['search_query']].values]
    e = item_emb[positions]
    locs = item_loc[positions]
    n7, n8 = [], []
    for start in range(0, len(qdf), 500):
        sims = q[start:start + 500] @ e.T
        same = locs[None, :] == qdf['search_location_id'].values[start:start + 500, None]
        n7.append(((sims > 0.7) & same).sum(axis=1))
        n8.append(((sims > 0.8) & same).sum(axis=1))
    return np.concatenate(n7), np.concatenate(n8)


bench_pos = np.arange(n_bench_items)
old_val_pos = np.flatnonzero((np.arange(len(all_corpus)) < n_bench_items) | all_corpus['item_id'].isin(val_relevant_ids).values)
rows = []
for name, qdf, pos in [('benchmark', queries, bench_pos), ('валидация, первая версия', val_ab, old_val_pos)]:
    n7, n8 = close_items(qdf, pos)
    rows.append({'запросы': name, 'медиана, косинус > 0.7': np.median(n7), 'медиана, косинус > 0.8': np.median(n8), 'среднее, косинус > 0.8': n8.mean().round(1)})
pd.DataFrame(rows)

У запросов benchmark рядом (в той же локации и по смыслу) заметно больше конкурентов, чем у валидационных. При этом если считать похожие объявления по всему корпусу без учета локации, разницы почти нет. То есть корпус benchmark, похоже, собран вокруг самих запросов: в нем много объявлений, которые подходили бы под запрос и находятся в том же городе (например, все, что показывалось в выдаче по этому запросу). Нужное объявление приходится отличать от них, а в первой валидации таких конкурентов почти не было, поэтому локация решала почти все.

Исправление: для каждого валидационного запроса добавляю в корпус "похожие" объявления:

- берутся только объявления из валидационной части, которых нет в обучающей части (как и в benchmark, где почти все объявления новые);
- только из локаций, куда при таком поиске реально уходят пользователи: та же локация или локации, куда по статистике обучающей части уходит хотя бы 5% выбора;
- 30 лучших по BM25 и 30 лучших по эмбеддингам, кроме релевантных объявлений самого запроса.

Число 30 подобрано так, чтобы плотность похожих объявлений стала близка к benchmark. Проверка калибровки - строки "первая версия" и "вторая версия" в таблице раздела 9: это веса двух уже отправленных решений, на платформе они получили 0.877 и 0.882. Если валидация устроена правильно, на ней должны получаться близкие числа.

In [ ]:
def bm25_matrix(texts, k1=1.2, b=0.75):
    vectorizer = CountVectorizer(analyzer=str.split, dtype=np.float32)
    tf = vectorizer.fit_transform(texts).tocsr()
    n_docs = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    doc_len = np.asarray(tf.sum(axis=1)).ravel()
    avg_len = max(doc_len.mean(), 1.0)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    x = tf.data
    tf.data = (x * (k1 + 1) / (x + k1 * (1 - b + b * doc_len[rows] / avg_len)) * idf[tf.indices]).astype(np.float32)
    return tf.T.tocsr(), vectorizer.vocabulary_


def query_matrix(texts, vocabulary):
    return CountVectorizer(analyzer=str.split, vocabulary=vocabulary, binary=True, dtype=np.float32).transform(texts)

In [ ]:
K_NEG = 30
t = time.time()
counts = train_part.groupby(['search_location_id', 'item_location_id']).size().reset_index(name='n')
counts['p'] = counts['n'] / counts.groupby('search_location_id')['n'].transform('sum')
allowed_locs = counts[counts['p'] >= 0.05].groupby('search_location_id')['item_location_id'].apply(set).to_dict()

cold_pos = np.flatnonzero(all_corpus['item_id'].isin(cold_ids).values)
cold_item_ids = all_corpus['item_id'].values[cold_pos]
cold_index = pd.Series(np.arange(len(cold_pos)), index=cold_item_ids)
cold_loc = item_loc[cold_pos]
cold_emb = item_emb[cold_pos]
cold_bm25, cold_vocab = bm25_matrix(all_corpus['full_st'].values[cold_pos])

negatives = set()
for start in range(0, len(val_ab), 200):
    batch = val_ab.iloc[start:start + 200]
    bm = (query_matrix(batch['query_st'], cold_vocab) @ cold_bm25).toarray()
    em = query_emb[query_emb_pos.loc[batch['search_query']].values] @ cold_emb.T
    for row in range(len(batch)):
        q = batch.iloc[row]
        locs = allowed_locs.get(q['search_location_id'], set()) | {q['search_location_id']}
        ok = np.isin(cold_loc, list(locs))
        ok[cold_index.reindex(q['relevant']).dropna().astype(int).values] = False
        for scores in [np.where(ok & (bm[row] > 0), bm[row], -np.inf), np.where(ok, em[row], -np.inf)]:
            top = np.argsort(-scores, kind='stable')[:K_NEG]
            negatives.update(cold_pos[top[np.isfinite(scores[top])]].tolist())

keep = np.zeros(len(all_corpus), dtype=bool)
keep[:n_bench_items] = True
keep[all_corpus['item_id'].isin(val_relevant_ids).values] = True
keep[sorted(negatives)] = True
val_corpus = all_corpus[keep].reset_index(drop=True)
print('добавлено похожих объявлений:', len(negatives), '| корпус для валидации:', len(val_corpus), '| за', round(time.time() - t), 'сек')

n7, n8 = close_items(val_ab, np.flatnonzero(keep))
print('новая валидация: медиана косинус > 0.7:', np.median(n7), '| косинус > 0.8:', np.median(n8), '| среднее косинус > 0.8:', n8.mean().round(1))

## 7. Дообучение моделей эмбеддингов

На новой валидации стало видно, что важно не только найти объявления в нужном городе, но и точнее понять, какое из похожих объявлений подходит под запрос. Готовые модели эмбеддингов обучались на общих текстах, а в train есть почти полмиллиона примеров "запрос - выбранное объявление". Дообучаю на этих парах две модели:

- `rubert-tiny-turbo` - та же, что в разделе 5;
- `sergeyzh/rubert-mini-frida` - модель чуть больше (32 млн параметров, тоже 312-мерные эмбеддинги), дистиллированная из FRIDA. В моих экспериментах она без дообучения почти догоняла дообученную tiny-модель, а после дообучения две модели вместе работали лучше, чем каждая по отдельности. У нее есть служебные префиксы `search_query: ` и `search_document: `, с которыми она обучалась, поэтому я добавляю их к текстам и при дообучении, и при расчете эмбеддингов.

Как устроено обучение (одинаково для обеих моделей):

- для каждого текста запроса беру до 3 разных выбранных объявлений; текст объявления - заголовок + первые 100 символов описания;
- в батче 64 пары. Для каждого запроса "правильное" объявление - его пара, а остальные 63 объявления батча считаются неправильными. Модель учится, чтобы косинус запроса со своей парой был выше, чем с чужими (это обычная схема для таких моделей, кросс-энтропия по матрице косинусов);
- в одном батче нет одинаковых текстов запросов, чтобы не наказывать модель за правильный ответ;
- одна эпоха, AdamW, lr = 5e-5 с разогревом и линейным спадом, все случайности зафиксированы, включен детерминированный режим PyTorch.

Модели обучаются два раза:

- для валидации - только на парах из обучающей части (около 87 тыс. пар), иначе была бы утечка;
- для финального ответа - на парах из всего train (около 123 тыс. пар), так же, как вся остальная статистика для финального ответа считается по всему train. Это важно: я проверял, что tiny-модель, дообученная на 60% пар обучающей части, на валидации хуже примерно на 0.3-0.4 пункта. К тому же тексты запросов benchmark, которые встречались в train, тогда все попадают в обучение, как и знакомые тексты на валидации.

На CPU tiny-модель учится около 17 минут на обучающей части и 25 минут на всем train, mini-frida - около 35 и 50 минут. Готовые модели сохраняются в `cache/`, при повторном запуске обучение пропускается.

In [ ]:
def make_pairs(df):
    pairs = df.drop_duplicates(['search_query', 'item_id']).sample(frac=1, random_state=SEED)
    pairs['r'] = pairs.groupby('search_query').cumcount()
    pairs = pairs[pairs['r'] < 3].copy()
    pairs['text'] = pairs['item_title_raw'] + '. ' + pairs['item_description_raw'].str.slice(0, 100)
    batches = []
    for r, group in pairs.groupby('r'):
        group = group.sample(frac=1, random_state=r)
        for start in range(0, len(group) - 63, 64):
            batches.append(group.iloc[start:start + 64])
    random.Random(SEED).shuffle(batches)
    assert all(b['search_query'].is_unique for b in batches)
    return pairs, batches


part_pairs, part_batches = make_pairs(train_part)
full_pairs, full_batches = make_pairs(train)
print('пар из обучающей части:', len(part_pairs), '| батчей:', len(part_batches))
print('пар из всего train:', len(full_pairs), '| батчей:', len(full_batches))

In [ ]:
def encode_for_training(model, texts, max_len):
    model.max_seq_length = max_len
    features = model.preprocess(texts)
    return torch.nn.functional.normalize(model(features)['sentence_embedding'], dim=-1)


def fine_tune(model_name, batches, path, query_prompt='', doc_prompt=''):
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    model = SentenceTransformer(model_name, device='cpu')
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    n_steps = len(batches)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(1.0, (step + 1) / 100) * max(0.0, 1 - step / n_steps))
    model.train()
    for step, batch in enumerate(batches):
        q = encode_for_training(model, [query_prompt + x for x in batch['search_query']], 32)
        d = encode_for_training(model, [doc_prompt + x for x in batch['text']], 48)
        sim = q @ d.T * 20
        labels = torch.arange(len(batch))
        loss = (torch.nn.functional.cross_entropy(sim, labels) + torch.nn.functional.cross_entropy(sim.T, labels)) / 2
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        if step % 500 == 0:
            print('шаг', step, 'из', n_steps, '| loss', round(loss.item(), 3))
    model.eval()
    model.save(path)
    torch.use_deterministic_algorithms(False)


def get_fine_tuned(model_name, pairs, batches, query_prompt='', doc_prompt=''):
    key = hashlib.md5(pd.util.hash_pandas_object(pairs[['search_query', 'text']], index=False).values.tobytes()).hexdigest()[:12]
    path = os.path.join(CACHE_DIR, 'ft_' + model_name.split('/')[-1] + '_' + key)
    t = time.time()
    if not os.path.exists(path):
        fine_tune(model_name, batches, path, query_prompt, doc_prompt)
    print(path, 'готова за', round(time.time() - t), 'сек')
    return SentenceTransformer(path, device='cpu'), os.path.basename(path)


ft_model, ft_name = get_fine_tuned(EMB_MODEL_NAME, part_pairs, part_batches)
frida_model, frida_name = get_fine_tuned(FRIDA_MODEL_NAME, part_pairs, part_batches, FRIDA_QUERY_PROMPT, FRIDA_DOC_PROMPT)

In [ ]:
t = time.time()
ft_texts = (val_corpus['item_title_raw'] + '. ' + val_corpus['item_description_raw'].str.slice(0, 100)).tolist()
ft_item_emb = encode(ft_model, ft_texts, ft_name + '_items', 48)
ft_query_emb = encode(ft_model, all_query_texts, ft_name + '_queries', 32)
frida_item_emb = encode(frida_model, [FRIDA_DOC_PROMPT + x for x in ft_texts], frida_name + '_items', 48)
frida_query_emb = encode(frida_model, [FRIDA_QUERY_PROMPT + x for x in all_query_texts], frida_name + '_queries', 32)
print(ft_item_emb.shape, frida_item_emb.shape, 'за', round(time.time() - t), 'сек')

## 8. Сигналы для скоринга

Для каждого запроса считаю скор для всех объявлений корпуса сразу (матрицами, батчами по 200 запросов) и беру 50 лучших. Все, что считается по train, собирается функцией `build_index`: на валидации ей передается только обучающая часть и эмбеддинги моделей, дообученных на ней, а для финального ответа - весь train и модели, дообученные на всем train.

Сигналы:

1. `bm_full` - BM25 по заголовку + описанию + параметрам (стеммированные слова).
2. `bm_title` - BM25 только по заголовку.
3. `loc` - вероятность локации объявления при локации поиска, посчитанная по train: P(item_location | search_location). В скор идет её логарифм. Для локации поиска, которой нет в train, строка нулевая, и этот сигнал просто одинаковый для всех объявлений.
4. `dist` - расстояние (км) от "центра" локации поиска (медиана координат выбранных объявлений в train) до координат объявления.
5. `nl` - "нелокальность" запроса: по 20 похожим запросам из train смотрю, какая доля выборов была в другом месте (маловероятная локация или дальше 100 км). Для запросов вроде "база отдыха", "геологические изыскания", "аренда авто в стамбуле" это заметная доля. У таких запросов штраф за локацию ослабляется.
6. `flt` - фильтры: из `search_infm_params_text` достаю "Вид услуги ..." и "Тип услуги ..." и проверяю, есть ли они в `item_infm_params_text`. +1 если все совпали, -1 если нет, 0 если фильтров нет.
7. `mc` - подкатегория: по тем же 20 похожим запросам из train смотрю, из каких `item_microcat_id` там выбирали объявления.
8. `exp` - расширение запроса: заголовки объявлений, выбранных по похожим запросам, сравниваются с заголовками корпуса (TF-IDF косинус). Так находятся синонимы.
9. `emb` - косинус эмбеддингов готовой модели.
10. `emb_ft` - косинус эмбеддингов дообученной tiny-модели.
11. `emb_frida` - косинус эмбеддингов дообученной mini-frida.
12. `cat` - совпадает ли категория объявления с `search_category` запроса. Если запрос без категории (`search_category = 0`), подходит любая категория. Почему так - ниже.

Итоговый скор - взвешенная сумма этих сигналов, веса подбираются ниже по Recall@50.

Проверка про `search_category = 0`. В train почти все запросы из категории 114, и там выбирали только объявления из 114. Но в benchmark 9% запросов без категории. Смотрю, есть ли у них в корпусе очень похожие объявления (косинус больше 0.8) из других категорий в той же локации, и сравниваю с обычными запросами.

In [ ]:
non_services = np.flatnonzero(items['item_category_id'].values != 114)
titles_bench = items['item_title_raw'].values
rows, examples = [], []
for category in [114, 0]:
    qdf = queries[queries['search_category'] == category]
    sims = query_emb[query_emb_pos.loc[qdf['search_query']].values] @ item_emb[non_services].T
    same = qdf['search_location_id'].values[:, None] == item_loc[non_services][None, :]
    close = (sims > 0.8) & same
    rows.append({'search_category': category, 'запросов': len(qdf), 'доля запросов с близким объявлением не из 114': close.any(axis=1).mean().round(3)})
    if category == 0:
        for k in np.flatnonzero(close.any(axis=1))[:6]:
            j = non_services[np.argmax(np.where(same[k], sims[k], -1))]
            examples.append({'запрос': qdf['search_query'].values[k], 'объявление': titles_bench[j], 'категория': items['item_category_id'].values[j]})
display(pd.DataFrame(rows))
pd.DataFrame(examples)

У запросов без категории такие объявления находятся в 4-5 раз чаще, и они явно подходят под запрос ("надувные шарики" - "Надувные шары" из категории 30). Похоже, корпус собирали и вокруг этих запросов, и нужное объявление может быть не из услуг. На валидации это не проверить (в train таких запросов почти нет), поэтому делаю самое простое правило: бонус `cat` дается, если категория объявления совпадает с категорией поиска, а для поиска без категории он есть у всех объявлений. Для запросов из 114 это ровно то же, что раньше, на валидации ничего не меняется.

In [ ]:
def to_float(values):
    return pd.Series(values).astype(float).values


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

In [ ]:
all_params = pd.concat([train['item_infm_params_text'], items['item_infm_params_text']])
stop_words = r'(?: Место оказания| Тип услуги| Вид услуги| Тип стоимости|$)'
filter_values = {}
for key in ['Вид услуги', 'Тип услуги']:
    counts = all_params.str.extract(key + r' (.*?)' + stop_words)[0].value_counts()
    filter_values[key] = sorted([v for v in counts[counts >= 5].index if v and 'Место' not in v], key=len, reverse=True)


def parse_filters(params_text):
    found = []
    for key, values in filter_values.items():
        for v in values:
            if key + ' ' + v in params_text:
                found.append(key + ' ' + v)
                break
    return found


print({k: len(v) for k, v in filter_values.items()})
for text in ['', 'Вид услуги', 'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье', 'Рейтинг пользователя 4 звезды и выше']:
    print(repr(text), '->', parse_filters(text))

In [ ]:
sample = train.sample(50000, random_state=SEED)
sample_filters = sample['search_infm_params_text'].map({t: parse_filters(t) for t in sample['search_infm_params_text'].unique()})
has_filters = sample_filters.str.len() > 0
match = [all(f in p for f in fs) for fs, p in zip(sample_filters[has_filters], sample.loc[has_filters, 'item_infm_params_text'])]
print('строк с фильтрами:', round(has_filters.mean(), 3), '| фильтры совпали с параметрами объявления:', round(np.mean(match), 3))

In [ ]:
def build_index(train_df, corpus, embeddings, k_neighbors=20):
    index = {'corpus': corpus, 'item_ids': corpus['item_id'].to_numpy(dtype=object), 'embeddings': embeddings, 'k': k_neighbors}
    index['bm_full'], index['vocab_full'] = bm25_matrix(corpus['full_st'].values)
    index['bm_title'], index['vocab_title'] = bm25_matrix(corpus['title_st'].values, b=0.5)

    search_locs = pd.Index(sorted(train_df['search_location_id'].unique()))
    item_locs = pd.Index(sorted(set(train_df['item_location_id']) | set(corpus['item_location_id'])))
    counts = train_df.groupby(['search_location_id', 'item_location_id']).size().reset_index(name='n')
    loc_prior = np.zeros((len(search_locs) + 1, len(item_locs)), dtype=np.float32)
    loc_prior[search_locs.get_indexer(counts['search_location_id']), item_locs.get_indexer(counts['item_location_id'])] = counts['n'].values
    loc_prior = loc_prior / np.maximum(loc_prior.sum(axis=1, keepdims=True), 1)
    index['search_locs'] = search_locs
    index['loc_prior'] = loc_prior
    index['item_loc_pos'] = item_locs.get_indexer(corpus['item_location_id'])

    lat = to_float(corpus['item_latitude'])
    lon = to_float(corpus['item_longitude'])
    loc_center = pd.DataFrame({'loc': corpus['item_location_id'].values, 'lat': lat, 'lon': lon}).groupby('loc')[['lat', 'lon']].median()
    index['item_lat'] = np.where(np.isnan(lat), loc_center['lat'].reindex(corpus['item_location_id']).values, lat)
    index['item_lon'] = np.where(np.isnan(lon), loc_center['lon'].reindex(corpus['item_location_id']).values, lon)
    click_lat = to_float(train_df['item_latitude'])
    click_lon = to_float(train_df['item_longitude'])
    clicks = pd.DataFrame({'loc': train_df['search_location_id'].values, 'lat': click_lat, 'lon': click_lon})
    index['search_center'] = clicks.groupby('loc')[['lat', 'lon']].median()

    pair_p = loc_prior[search_locs.get_indexer(train_df['search_location_id']), item_locs.get_indexer(train_df['item_location_id'])]
    center = index['search_center'].reindex(train_df['search_location_id'])
    click_km = np.nan_to_num(haversine(center['lat'].values, center['lon'].values, click_lat, click_lon), nan=0.0)
    train_df = train_df.assign(is_far=((pair_p < 0.01) | (click_km > 100)).astype(np.float32))

    index['city_share'] = train_df['search_location_id'].value_counts(normalize=True)
    past = train_df.groupby('search_query').agg(titles=('title_st', ' '.join), microcats=('item_microcat_id', list), far=('is_far', 'mean')).reset_index()
    index['past_emb'] = query_emb[query_emb_pos.loc[past['search_query']].values]
    index['past_far'] = past['far'].values.astype(np.float32)
    title_tfidf = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, min_df=2, dtype=np.float32)
    title_tfidf.fit(corpus['title_st'])
    index['title_vecs'] = normalize(title_tfidf.transform(corpus['title_st'])).T.tocsr()
    index['past_titles'] = normalize(title_tfidf.transform(past['titles']))

    microcats = pd.Index(sorted(train_df['item_microcat_id'].unique()))
    rows = np.repeat(np.arange(len(past)), past['microcats'].str.len())
    cols = microcats.get_indexer(np.concatenate(past['microcats'].values))
    mc = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(past), len(microcats)))
    index['past_microcats'] = normalize(mc, norm='l1')
    item_mc_pos = microcats.get_indexer(corpus['item_microcat_id'])
    item_mc_pos[item_mc_pos < 0] = len(microcats)
    index['item_mc_pos'] = item_mc_pos

    index['item_category'] = corpus['item_category_id'].values
    index['filter_cache'] = {}
    return index

In [ ]:
def filter_scores(index, params_texts):
    result = np.zeros((len(params_texts), len(index['item_ids'])), dtype=np.float32)
    for row, text in enumerate(params_texts):
        filters = parse_filters(text)
        if not filters:
            continue
        ok = np.ones(len(index['item_ids']), dtype=bool)
        for f in filters:
            if f not in index['filter_cache']:
                index['filter_cache'][f] = index['corpus']['item_infm_params_text'].str.contains(f, regex=False).values
            ok &= index['filter_cache'][f]
        result[row] = np.where(ok, 1, -1)
    return result


def compute_signals(index, batch):
    signals = {}
    signals['bm_full'] = (query_matrix(batch['query_st'], index['vocab_full']) @ index['bm_full']).toarray()
    signals['bm_title'] = (query_matrix(batch['query_st'], index['vocab_title']) @ index['bm_title']).toarray()

    q_pos = query_emb_pos.loc[batch['search_query']].values
    for name, (q_vectors, item_vectors) in index['embeddings'].items():
        signals[name] = q_vectors[q_pos] @ item_vectors.T
    q_emb = query_emb[q_pos]

    k = index['k']
    sims = q_emb @ index['past_emb'].T
    nearest = np.argpartition(-sims, k, axis=1)[:, :k]
    weights = np.clip(np.take_along_axis(sims, nearest, axis=1), 0, None) ** 2
    rows = np.repeat(np.arange(len(batch)), k)
    neighbors = sp.csr_matrix((weights.ravel(), (rows, nearest.ravel())), shape=sims.shape)
    signals['exp'] = (normalize(neighbors @ index['past_titles']) @ index['title_vecs']).toarray()
    mc = normalize(neighbors @ index['past_microcats'], norm='l1').toarray()
    mc = np.hstack([mc, np.zeros((len(batch), 1), dtype=np.float32)])
    signals['mc'] = mc[:, index['item_mc_pos']]
    nl = (weights * index['past_far'][nearest]).sum(axis=1) / np.maximum(weights.sum(axis=1), 1e-9)
    signals['nl'] = np.broadcast_to(nl[:, None].astype(np.float32), signals['emb'].shape)

    loc_pos = index['search_locs'].get_indexer(batch['search_location_id'])
    loc_pos[loc_pos < 0] = len(index['search_locs'])
    signals['loc'] = index['loc_prior'][loc_pos][:, index['item_loc_pos']]
    center = index['search_center'].reindex(batch['search_location_id'])
    dist = haversine(center['lat'].values[:, None], center['lon'].values[:, None], index['item_lat'][None, :], index['item_lon'][None, :])
    signals['dist'] = np.nan_to_num(dist, nan=0.0).astype(np.float32)

    signals['flt'] = filter_scores(index, batch['search_infm_params_text'].values)
    search_category = batch['search_category'].values[:, None]
    signals['cat'] = ((index['item_category'][None, :] == search_category) | (search_category == 0)).astype(np.float32)
    return signals


def combine(s, w):
    score = w['bm_full'] * s['bm_full'] + w['bm_title'] * s['bm_title'] + w['exp'] * s['exp']
    score = score + w['emb'] * s['emb'] + w['emb_ft'] * s['emb_ft'] + w['emb_frida'] * s['emb_frida']
    local = 1 - w['loc_local'] * s['nl']
    score = score + w['loc'] * local * np.log(s['loc'] + w['loc_eps']) - w['dist'] * np.log1p(s['dist'] / w['dist_scale'])
    score = score + w['mc'] * np.log(s['mc'] + w['mc_eps']) + w['flt'] * s['flt'] + w['cat'] * s['cat']
    return score


def top_k(score, k=50):
    k = min(k, score.shape[1])
    part = np.argpartition(-score, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(score, part, axis=1), axis=1, kind='stable')
    return np.take_along_axis(part, order, axis=1)

Функция прогона: для батча считает сигналы один раз, а дальше для каждого набора весов берет топ-50. Если передать `pool_weights`, она еще сохраняет "пул кандидатов" для подбора весов: объединение топ-300 по нескольким разным наборам весов вместе со значениями всех сигналов. Потом веса перебираются уже только внутри пула, это в сотни раз быстрее, чем пересчитывать весь корпус.

In [ ]:
SIGNALS = ['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'emb_frida', 'mc', 'nl', 'loc', 'dist', 'flt', 'cat']


def run(index, qdf, weight_sets, pool_weights=None, pool_size=300, batch_size=200):
    predictions = {name: [] for name in weight_sets}
    pool = []
    for start in range(0, len(qdf), batch_size):
        batch = qdf.iloc[start:start + batch_size]
        s = compute_signals(index, batch)
        for name, w in weight_sets.items():
            top = top_k(combine(s, w))
            predictions[name].extend([list(index['item_ids'][row]) for row in top])
        if pool_weights is not None:
            cand = np.concatenate([top_k(combine(s, w), pool_size) for w in pool_weights], axis=1)
            for row in range(len(batch)):
                c = np.unique(cand[row])
                part = pd.DataFrame({name: s[name][row, c] for name in SIGNALS})
                part['q'] = start + row
                part['item_id'] = index['item_ids'][c]
                pool.append(part)
    if pool_weights is not None:
        return predictions, pd.concat(pool, ignore_index=True)
    return predictions

## 9. Пошаговая проверка на валидации

Собираю индекс по обучающей части и прогоняю выборки A и B. Начальные веса выставлены вручную по порядку величин сигналов (BM25 порядка 5-20, косинусы 0-1, логарифм вероятности локации от -10 до 0). Каждая следующая строка добавляет один сигнал. Строки "первая версия" и "вторая версия" - веса, с которыми были отправлены первые два ответа (0.877 и 0.882 на платформе). Вторая версия отличалась от первой дообученной tiny-моделью и нелокальностью запроса.

Важно: веса в этой таблице выставлены вручную и не согласованы между собой, поэтому последние шаги (дообученные эмбеддинги и нелокальность) тут могут даже ухудшать результат. Реальный вклад каждого сигнала видно в таблице "без ..." после подбора весов (раздел 10).

In [ ]:
t = time.time()
val_embeddings = {'emb': (query_emb, item_emb[keep]), 'emb_ft': (ft_query_emb, ft_item_emb), 'emb_frida': (frida_query_emb, frida_item_emb)}
val_index = build_index(train_part, val_corpus, val_embeddings)
print('индекс для валидации собран за', round(time.time() - t), 'сек')

zero = dict(bm_full=0, bm_title=0, exp=0, emb=0, emb_ft=0, emb_frida=0, loc=0, loc_eps=1e-4, loc_local=0, dist=0, dist_scale=30, mc=0, mc_eps=1e-3, flt=0, cat=0)
steps = {}
steps['1. BM25 по тексту'] = dict(zero, bm_full=1)
steps['2. + BM25 по заголовку'] = dict(steps['1. BM25 по тексту'], bm_title=0.5)
steps['3. + локация'] = dict(steps['2. + BM25 по заголовку'], loc=2)
steps['4. + фильтры'] = dict(steps['3. + локация'], flt=2)
steps['5. + подкатегория'] = dict(steps['4. + фильтры'], mc=1)
steps['6. + эмбеддинги'] = dict(steps['5. + подкатегория'], emb=10)
steps['7. + расширение запроса'] = dict(steps['6. + эмбеддинги'], exp=5)
steps['8. + расстояние'] = dict(steps['7. + расширение запроса'], dist=0.5)
steps['9. + дообученные эмбеддинги'] = dict(steps['8. + расстояние'], emb_ft=10)
steps['10. + нелокальность запроса'] = dict(steps['9. + дообученные эмбеддинги'], loc_local=1)
steps['11. + дообученная frida'] = dict(steps['10. + нелокальность запроса'], emb_frida=10)
steps['первая версия (отправлена, 0.877)'] = dict(zero, bm_full=1, exp=5, emb=40, loc=2, loc_eps=3e-5, dist=1, dist_scale=30, mc=0.5, mc_eps=1e-3, flt=5)
steps['вторая версия (отправлена, 0.882)'] = dict(zero, bm_full=1, bm_title=0.25, exp=2.5, emb=10, emb_ft=20, loc=2.5, loc_eps=3e-5, loc_local=2,
                                                  dist=2, dist_scale=10, mc=0.25, mc_eps=1e-4, flt=5, cat=2)
pool_weights = [
    steps['3. + локация'],
    dict(zero, emb=20, loc=2),
    dict(zero, emb_ft=20, loc=2, dist=1),
    dict(zero, exp=20, loc=2, flt=2),
    steps['первая версия (отправлена, 0.877)'],
    dict(steps['первая версия (отправлена, 0.877)'], emb_ft=10, dist=2, dist_scale=10, loc_local=1),
    dict(zero, emb_frida=20, loc=2, dist=1),
    steps['вторая версия (отправлена, 0.882)'],
    dict(steps['вторая версия (отправлена, 0.882)'], emb_frida=10),
]

t = time.time()
val_pred, pool = run(val_index, val_ab, steps, pool_weights=pool_weights)
print('прогон валидации занял', round(time.time() - t), 'сек, строк в пуле:', len(pool))

is_a = (val_ab['part'] == 'A').values
rows = []
for name, pred in val_pred.items():
    rows.append({'шаг': name,
                 'Recall@50 A': recall_at_50([p for p, f in zip(pred, is_a) if f], val_ab.loc[is_a, 'relevant']),
                 'Recall@50 B': recall_at_50([p for p, f in zip(pred, is_a) if not f], val_ab.loc[~is_a, 'relevant'])})
pd.DataFrame(rows).round(4)

## 10. Подбор весов по валидации

Перебор по координатам: беру один вес, пробую несколько значений, оставляю лучшее по Recall@50 на выборке A, перехожу к следующему. Три полных прохода, старт с весов второй версии. Выборку B при подборе не трогаю и смотрю на нее только в конце.

Метрика внутри пула считается так же, как обычная: сортирую кандидатов запроса по скору, беру 50 лучших и делю число найденных релевантных на число всех релевантных запроса (включая те, что в пул не попали).

In [ ]:
relevant_sets = [set(r) for r in val_ab['relevant']]
pool['rel'] = [item in relevant_sets[q] for q, item in zip(pool['q'], pool['item_id'])]
n_relevant = val_ab['relevant'].str.len().values
pool_q = pool['q'].values
pool_rel = pool['rel'].values
pool_signals = {name: pool[name].values for name in SIGNALS}
print('доля релевантных, которые вообще попали в пул:', round(pool['rel'].sum() / n_relevant.sum(), 4))


def pool_recall(w):
    score = combine(pool_signals, w)
    order = np.lexsort((-score, pool_q))
    q_sorted = pool_q[order]
    starts = np.r_[0, np.flatnonzero(np.diff(q_sorted)) + 1]
    rank = np.arange(len(order)) - np.repeat(starts, np.diff(np.r_[starts, len(order)]))
    hits = np.bincount(q_sorted[(rank < 50) & pool_rel[order]], minlength=len(val_ab))
    recall = hits / n_relevant
    return recall[is_a].mean(), recall[~is_a].mean()


grid = {
    'bm_full': [0.5, 1, 1.5, 2],
    'bm_title': [0, 0.25, 0.5, 1],
    'exp': [0, 2.5, 5, 10, 15],
    'emb': [0, 10, 20, 30, 40, 60],
    'emb_ft': [0, 5, 10, 20, 30, 40],
    'emb_frida': [0, 5, 10, 20, 30],
    'loc': [1, 1.5, 2, 2.5, 3],
    'loc_eps': [1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
    'loc_local': [0, 0.5, 1, 1.5, 2],
    'dist': [0, 0.5, 1, 2, 3, 5],
    'dist_scale': [3, 10, 30, 100, 300],
    'mc': [0, 0.25, 0.5, 1, 2],
    'mc_eps': [1e-4, 1e-3, 1e-2],
    'flt': [0, 1, 2, 3, 5, 8],
    'cat': [0, 1, 2],
}

best_w = dict(steps['вторая версия (отправлена, 0.882)'])
best_score = pool_recall(best_w)[0]
print('старт: A =', round(best_score, 4))
for iteration in range(3):
    for name, values in grid.items():
        for v in values:
            w = dict(best_w, **{name: v})
            score = pool_recall(w)[0]
            if score > best_score + 1e-6:
                best_score, best_w = score, w
    print(f'проход {iteration + 1}: A = {best_score:.4f}')
print(best_w)

In [ ]:
a, b = pool_recall(best_w)
rows = [{'вариант': 'все сигналы', 'A': a, 'B': b}]
for name in ['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'emb_frida', 'mc', 'loc', 'loc_local', 'dist', 'flt']:
    a, b = pool_recall(dict(best_w, **{name: 0}))
    rows.append({'вариант': 'без ' + name, 'A': a, 'B': b})
pd.DataFrame(rows).round(4)

Проверка на отложенной выборке B уже полным прогоном по всему корпусу (без пула), с подобранными весами и с весами двух отправленных версий.

In [ ]:
t = time.time()
pred_b = run(val_index, val_b, {'best': best_w, 'first': steps['первая версия (отправлена, 0.877)'], 'second': steps['вторая версия (отправлена, 0.882)']})
print('Recall@50 на B, первая версия:', round(recall_at_50(pred_b['first'], val_b['relevant']), 4))
print('Recall@50 на B, вторая версия:', round(recall_at_50(pred_b['second'], val_b['relevant']), 4))
print('Recall@50 на B, подобранные веса:', round(recall_at_50(pred_b['best'], val_b['relevant']), 4), '| время', round(time.time() - t), 'сек')

## 10.1. Переранжирование градиентным бустингом

Взвешенная сумма не умеет учитывать сочетания сигналов: например, что для запроса без фильтров важнее текст, а для крупного города расстояние. Поэтому поверх нее ставлю второй этап, как в обычном поисковом каскаде:

- по взвешенной сумме беру 150 лучших кандидатов на запрос (нужное объявление есть среди них примерно в 95% случаев);
- для каждой пары "запрос - кандидат" собираю признаки: все сигналы, сам скор взвешенной суммы и место кандидата по нему, отставание кандидата от лучшего в этом запросе по каждому текстовому сигналу, число слов в запросе, есть ли фильтры, доля запросов из этого города в train;
- `HistGradientBoostingClassifier` из sklearn учится отличать релевантные объявления от остальных, в ответ идут 50 кандидатов с самой большой вероятностью.

Бустинг учу на выборке A, проверяю на B. Для финального ответа учу на A и B вместе. Первая попытка бустинга (в прошлой версии ноутбука) переобучалась, потому что он заменял взвешенную сумму целиком. Здесь он получает ее скор как признак и только поправляет порядок, и так работает заметно лучше.

In [ ]:
RERANK_K = 150
GAP_SIGNALS = ['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'emb_frida', 'lin']
RERANK_FEATURES = (['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'emb_frida', 'log_mc', 'nl', 'log_loc', 'log_dist', 'flt', 'cat',
                    'n_words', 'has_filters', 'city_share', 'lin', 'rank'] + [name + '_gap' for name in GAP_SIGNALS])


def rerank_table(index, qdf, w):
    _, cand = run(index, qdf, {}, pool_weights=[w], pool_size=RERANK_K)
    cand['lin'] = combine({name: cand[name].values for name in SIGNALS}, w)
    cand['rank'] = cand.groupby('q')['lin'].rank(ascending=False, method='first')
    for name in GAP_SIGNALS:
        cand[name + '_gap'] = cand[name] - cand.groupby('q')[name].transform('max')
    cand['log_loc'] = np.log(cand['loc'] + 3e-5)
    cand['log_dist'] = np.log1p(cand['dist'])
    cand['log_mc'] = np.log(cand['mc'] + 1e-4)
    q = cand['q'].values
    cand['n_words'] = qdf['search_query'].str.split().str.len().values[q]
    cand['has_filters'] = (qdf['search_infm_params_text'] != '').values[q].astype(int)
    cand['city_share'] = np.log(index['city_share'].reindex(qdf['search_location_id']).fillna(0).values + 1e-6)[q]
    return cand


def top50_by(cand, column, n_queries):
    ordered = cand.sort_values(['q', column], ascending=[True, False])
    top = ordered.groupby('q').head(50)
    lists = top.groupby('q')['item_id'].apply(list)
    return [lists.get(i, []) for i in range(n_queries)]


def make_reranker():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                                          l2_regularization=1.0, random_state=SEED)


t = time.time()
val_cand = rerank_table(val_index, val_ab, best_w)
val_cand['rel'] = [item in relevant_sets[q] for q, item in zip(val_cand['q'], val_cand['item_id'])]
cand_is_a = is_a[val_cand['q'].values]
print('кандидатов:', len(val_cand), '| доля релевантных, попавших в топ-150: A', round(val_cand.loc[cand_is_a, 'rel'].sum() / n_relevant[is_a].sum(), 4),
      'B', round(val_cand.loc[~cand_is_a, 'rel'].sum() / n_relevant[~is_a].sum(), 4), '| за', round(time.time() - t), 'сек')

reranker = make_reranker()
reranker.fit(val_cand.loc[cand_is_a, RERANK_FEATURES], val_cand.loc[cand_is_a, 'rel'])
val_cand['prob'] = reranker.predict_proba(val_cand[RERANK_FEATURES])[:, 1]
print('итераций бустинга:', reranker.n_iter_)

rerank_pred = top50_by(val_cand, 'prob', len(val_ab))
linear_pred = top50_by(val_cand, 'lin', len(val_ab))
rows = []
for name, pred in [('взвешенная сумма', linear_pred), ('+ бустинг', rerank_pred)]:
    rows.append({'вариант': name,
                 'A (бустинг учился на ней)': recall_at_50([p for p, f in zip(pred, is_a) if f], val_ab.loc[is_a, 'relevant']),
                 'B': recall_at_50([p for p, f in zip(pred, is_a) if not f], val_ab.loc[~is_a, 'relevant'])})
pd.DataFrame(rows).round(4)

Какие признаки важнее всего для бустинга: перемешиваю значения одного признака на выборке B и смотрю, насколько падает Recall@50.

In [ ]:
b_cand = val_cand[~cand_is_a].copy()
b_relevant = val_ab.loc[~is_a, 'relevant'].tolist()
b_cand['q'] = b_cand['q'] - is_a.sum()
base_b = recall_at_50(top50_by(b_cand, 'prob', len(val_b)), b_relevant)
perm_rng = np.random.RandomState(SEED)
rows = []
for name in RERANK_FEATURES:
    shuffled = b_cand[RERANK_FEATURES].copy()
    shuffled[name] = perm_rng.permutation(shuffled[name].values)
    b_cand['perm'] = reranker.predict_proba(shuffled)[:, 1]
    rows.append({'признак': name, 'падение Recall@50 на B': base_b - recall_at_50(top50_by(b_cand, 'perm', len(val_b)), b_relevant)})
pd.DataFrame(rows).sort_values('падение Recall@50 на B', ascending=False).head(12).round(4)

Несколько примеров с выборки B: запрос, релевантное объявление и первые 5 кандидатов после бустинга.

In [ ]:
val_b['pred'] = pd.Series(rerank_pred[is_a.sum():], index=val_b.index)
val_b['recall'] = [len(set(p) & set(r)) / len(r) for p, r in zip(val_b['pred'], val_b['relevant'])]
print('Recall@50 на B после бустинга:', round(val_b['recall'].mean(), 4))
print('виденные тексты:', round(val_b.loc[val_b['seen'], 'recall'].mean(), 4), '| новые тексты:', round(val_b.loc[~val_b['seen'], 'recall'].mean(), 4))
print('кандидатов на запрос:', pd.Series([len(p) for p in val_b['pred']]).value_counts().to_dict())
print('запросов с дублями внутри ответа:', sum(len(p) != len(set(p)) for p in val_b['pred']))
titles = pd.Series(val_corpus['item_title_raw'].values, index=val_corpus['item_id'].values)
for _, row in val_b.sample(3, random_state=1).iterrows():
    print()
    print('ЗАПРОС:', row['search_query'], '| фильтры:', row['search_infm_params_text'] or '-', '| recall:', row['recall'])
    print('   нужно:', [titles[i] for i in row['relevant']])
    print('   топ-5:', [titles[i] for i in row['pred'][:5]])
val_b['n_words'] = val_b['search_query'].str.split().str.len().clip(upper=5)
val_b.groupby('n_words')['recall'].agg(['mean', 'size']).round(3)

Разбор промахов: для объявлений, которые не попали в топ-50, смотрю, насколько вероятна была их локация.

In [ ]:
missed = []
for _, row in val_b.iterrows():
    for item_id in row['relevant']:
        if item_id not in set(row['pred']):
            missed.append({'query': row['search_query'], 'search_location_id': row['search_location_id'], 'item_id': item_id})
missed = pd.DataFrame(missed)
missed = missed.merge(val_corpus[['item_id', 'item_title_raw', 'item_location_id']], on='item_id', how='left')
pair_counts = train_part.groupby(['search_location_id', 'item_location_id']).size()
search_counts = train_part.groupby('search_location_id').size()
pair_n = pair_counts.reindex(list(zip(missed['search_location_id'], missed['item_location_id']))).fillna(0).values
missed['p_loc'] = pair_n / search_counts.reindex(missed['search_location_id']).fillna(1).values
print('промахов:', len(missed), 'из', val_b['relevant'].str.len().sum())
print('доля промахов, где P(локация объявления | локация поиска) < 0.01:', round((missed['p_loc'] < 0.01).mean(), 3))
print('доля промахов, где такой пары локаций вообще не было в train:', round((missed['p_loc'] == 0).mean(), 3))
missed.sample(12, random_state=0)[['query', 'item_title_raw', 'p_loc']]

Выводы по валидации:

- Калибровка: на B первая отправленная версия получает 0.880 (на платформе 0.877), вторая 0.892 (на платформе 0.882). Валидация все еще немного завышает результат, но разница стала маленькой, и порядок версий тот же.
- Один BM25 дает около 0.39. Самый большой прирост по-прежнему дает локация (без нее 0.76 вместо 0.89). Когда рядом много похожих объявлений, важны расстояние и фильтры.
- Дообученная tiny-модель - самый сильный текстовый сигнал после BM25 (без нее 0.874 вместо 0.887).
- Повторный подбор весов на этой валидации изменил только вес `exp`: на A +0.1 пункта, на B -0.4 пункта по сравнению с весами второй версии. Для выборки из 3000 запросов это уровень шума, то есть веса взвешенной суммы уже почти не улучшаются. Вес frida во взвешенной сумме получился 0, но в бустинге она полезна (таблица важности признаков).
- Бустинг поверх взвешенной суммы дает самый большой прирост в этой версии: 0.902 на B против 0.887 у взвешенной суммы и 0.892 у весов второй версии. Самые важные признаки для него - место кандидата и отставание от лучшего кандидата по взвешенной сумме, дальше расстояние, отставание по BM25, размер города, локация и frida.
- На текстах, которые встречались в train, 0.928, на новых 0.887.
- Остаются два типа ошибок: запросы, под которые подходят сотни объявлений в одном городе, и объявления из "необычной" локации (около трети промахов).

## 11. Финальный ответ для benchmark

Теперь все считается по всему train: статистика в индексе и обе дообученные модели (обучаются заново на парах из всего train, раздел 7). Корпус - только `benchmark_items`, веса взвешенной суммы - подобранные выше. Бустинг переобучаю на выборках A и B вместе (больше примеров), с теми же настройками.

In [ ]:
t = time.time()
final_reranker = make_reranker()
final_reranker.fit(val_cand[RERANK_FEATURES], val_cand['rel'])
print('итераций бустинга:', final_reranker.n_iter_)

final_ft_model, final_ft_name = get_fine_tuned(EMB_MODEL_NAME, full_pairs, full_batches)
final_frida_model, final_frida_name = get_fine_tuned(FRIDA_MODEL_NAME, full_pairs, full_batches, FRIDA_QUERY_PROMPT, FRIDA_DOC_PROMPT)
bench_corpus = all_corpus.iloc[:n_bench_items].reset_index(drop=True)
assert (val_corpus['item_id'].iloc[:n_bench_items].values == bench_corpus['item_id'].values).all()
bench_texts = ft_texts[:n_bench_items]
final_embeddings = {
    'emb': (query_emb, item_emb[:n_bench_items]),
    'emb_ft': (encode(final_ft_model, all_query_texts, final_ft_name + '_queries', 32), encode(final_ft_model, bench_texts, final_ft_name + '_items', 48)),
    'emb_frida': (encode(final_frida_model, [FRIDA_QUERY_PROMPT + x for x in all_query_texts], final_frida_name + '_queries', 32),
                  encode(final_frida_model, [FRIDA_DOC_PROMPT + x for x in bench_texts], final_frida_name + '_items', 48)),
}
final_index = build_index(train, bench_corpus, final_embeddings)
bench_cand = rerank_table(final_index, queries, best_w)
bench_cand['prob'] = final_reranker.predict_proba(bench_cand[RERANK_FEATURES])[:, 1]
predictions = top50_by(bench_cand, 'prob', len(queries))
linear_predictions = top50_by(bench_cand, 'lin', len(queries))
print('предсказания готовы за', round(time.time() - t), 'сек')
same_share = np.mean([len(set(a) & set(b)) / 50 for a, b in zip(predictions, linear_predictions)])
print('доля общих объявлений в топ-50 с бустингом и без него:', round(same_share, 3))

answer = pd.DataFrame({
    'query_id': queries['query_id'].values,
    'answer': [' '.join(top50) for top50 in predictions],
})
print(answer.dtypes)
answer.to_csv('answer.csv', index=False)
answer.head()

## 12. Проверки ответа (чек-лист)

In [ ]:
check = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
with open('answer.csv', encoding='utf-8') as f:
    header = f.readline().strip()
lists = check['answer'].str.split(' ')
corpus_ids = set(items['item_id'])
answer_ids = set(i for lst in lists for i in lst)

checks = {
    'заголовок ровно query_id,answer': header == 'query_id,answer',
    'ровно две колонки': check.columns.tolist() == ['query_id', 'answer'],
    'строк столько же, сколько запросов': len(check) == len(queries),
    'все query_id из benchmark есть': set(check['query_id']) == set(queries['query_id']),
    'нет лишних query_id': set(check['query_id']) <= set(queries['query_id']),
    'нет повторов query_id': check['query_id'].is_unique,
    'query_id по 16 символов': check['query_id'].str.len().eq(16).all(),
    'не больше 50 item_id в строке': lists.str.len().le(50).all(),
    'нет пустых ответов': lists.str.len().ge(1).all(),
    'нет повторов item_id в строке': all(len(x) == len(set(x)) for x in lists),
    'все item_id есть в корпусе': answer_ids <= corpus_ids,
    'item_id через один пробел, только 0-9a-f': check['answer'].str.fullmatch(r'[0-9a-f]{16}( [0-9a-f]{16})*').all(),
    'файл совпадает с тем, что было в памяти': check['query_id'].tolist() == answer['query_id'].tolist() and check['answer'].tolist() == answer['answer'].tolist(),
}
for name, ok in checks.items():
    print('OK  ' if ok else 'FAIL', name)
assert all(checks.values())
print('item_id на запрос:', lists.str.len().value_counts().to_dict())

Пограничные случаи на запросах benchmark.

In [ ]:
bench = queries.copy()
bench['pred'] = pd.Series(predictions, index=bench.index)
bench['n_words'] = bench['search_query'].str.split().str.len()
bench_bm25 = (query_matrix(bench['query_st'], final_index['vocab_full']) @ final_index['bm_full']).tocsr()
bench['bm25_found'] = np.diff(bench_bm25.indptr)
bench['odd_symbols'] = bench['search_query'].str.contains(r'[^a-zа-яё0-9 ]', case=False)
empty_params = bench['search_infm_params_text'] == ''

print('пустой search_infm_params_text:', empty_params.sum(), 'запросов, кандидатов у них min =', bench.loc[empty_params, 'pred'].str.len().min())
print('запросы из одного слова:', (bench['n_words'] == 1).sum(), ', кандидатов min =', bench.loc[bench['n_words'] == 1, 'pred'].str.len().min())
print('запросы, где BM25 нашел меньше 50 объявлений:', (bench['bm25_found'] < 50).sum(), ', из них совсем 0:', (bench['bm25_found'] == 0).sum())
print('   у них кандидатов min =', bench.loc[bench['bm25_found'] < 50, 'pred'].str.len().min())
print('запросы с необычными символами:', bench['odd_symbols'].sum(), ', пример:', bench.loc[bench['odd_symbols'], 'search_query'].head(5).tolist())
print('локации поиска, которых нет в train:', (~bench['search_location_id'].isin(set(train['search_location_id']))).sum())

item_category = pd.Series(items['item_category_id'].values, index=items['item_id'].values)
for category in [114, 0]:
    preds = [p for p, c in zip(predictions, queries['search_category']) if c == category]
    answer_categories = item_category[[i for p in preds for i in p]]
    print(f'search_category = {category}: объявлений не из категории 114 в ответах', (answer_categories != 114).sum(), 'из', len(answer_categories))

text_key = pd.Series((items['item_title_raw'] + '|' + items['item_description_raw']).values, index=items['item_id'].values)
dup_slots = sum(len(p) - text_key[p].nunique() for p in predictions)
print('мест в топ-50, занятых копиями одинакового текста:', dup_slots, 'из', sum(len(p) for p in predictions))

По пограничным случаям:

- пустые фильтры и NaN в текстах не ломают пайплайн: пропуски заменены на пустые строки, у таких запросов просто нулевой сигнал `flt`;
- скор считается для всех объявлений корпуса, поэтому даже если по словам ничего не нашлось, 50 кандидатов набираются по локации, эмбеддингам и подкатегории. Повторов нет, потому что `argpartition` возвращает разные позиции корпуса, а `item_id` в корпусе уникальны;
- жесткой фильтрации по категории нет, так что редкие категории не выкидываются. Для запросов из 114 объявления других категорий получают меньший скор, а для запросов без категории штрафа нет (раздел 8);
- копии одинаковых объявлений занимают около 1% мест. На валидации пробовал оставлять только одну копию текста в топ-50, Recall@50 не изменился, поэтому оставил как есть;
- токенизатор на эмодзи, цифрах и латинице не падает (проверено в разделе 3).

## 13. Воспроизводимость

Случайность есть в сплите, в дообучении моделей и в бустинге. Везде зафиксирован `SEED = 42`, порядок батчей задается через `random.Random(SEED)`, на время обучения включен детерминированный режим PyTorch (`torch.use_deterministic_algorithms`), у бустинга задан `random_state`. BM25, TF-IDF и косинусы детерминированы. Ниже хэш файла, чтобы сравнить с отправленным `answer.csv` после повторного запуска.

In [ ]:
with open('answer.csv', 'rb') as f:
    print('md5 answer.csv:', hashlib.md5(f.read()).hexdigest())
print('весь ноутбук выполнился за', round((time.time() - start_time) / 60, 1), 'мин')

## 14. Итог

**Данные и признаки.** Из запроса: текст, локация, фильтры `search_infm_params_text`, категория поиска. Из объявления: заголовок, описание, параметры, подкатегория, категория, локация и координаты. Train используется для статистики (переходы "локация поиска -> локация объявления", подкатегории, заголовки и "нелокальность" по похожим запросам) и для дообучения двух моделей эмбеддингов (на валидации - только обучающая часть, для финального ответа - весь train). Цену, рейтинг и флаги не использовал: на валидации они ничего не давали (проверял число отзывов, наличие рейтинга, скрытый телефон, длину описания).

**Алгоритм.** Два этапа. Первый: для каждого запроса скор считается по всему корпусу как взвешенная сумма: BM25 (весь текст и заголовок), log P(локация объявления | локация поиска) с поправкой на нелокальность запроса, штраф за расстояние, совпадение фильтров, совпадение категории, log вероятности подкатегории по 20 похожим запросам из train, расширение запроса заголовками из train, косинус эмбеддингов готовой `rubert-tiny-turbo` и косинусы двух дообученных на парах из train моделей (`rubert-tiny-turbo` и `rubert-mini-frida`). Второй: 150 лучших кандидатов переранжирует градиентный бустинг (`HistGradientBoostingClassifier`), в ответ идут 50 лучших.

**Как проверял.** Свой сплит train по уникальным комбинациям `search_*` полей (40% в валидацию), две выборки по ~3000 запросов с таким же составом, как в benchmark (популярность текста в train, наличие фильтров, крупный город). Корпус для валидации - benchmark, релевантные объявления и "похожие" объявления рядом с каждым запросом (раздел 6), иначе валидация слишком легкая. Проверка калибровки - два уже отправленных решения (0.877 и 0.882 на платформе) получают на B 0.880 и 0.892. Веса и бустинг подбирались на A, итоговая оценка на B (разделы 9-10): взвешенная сумма 0.887, с бустингом 0.902.

**Ошибки и пограничные случаи.** Раздел 12: формат файла и типы ID, пустые фильтры, короткие запросы, запросы без совпадений по словам, редкие категории, запросы без категории, дубли объявлений, необычные символы. Разбор промахов в конце раздела 10.

**Что пробовал и не оставил.** TF-IDF по символьным n-граммам заголовка (ухудшал), поиск похожих запросов по символьным n-граммам (прироста нет), ограничение копий одинакового текста в топ-50 (прироста нет), признаки качества объявления (прироста нет), доля слов запроса в заголовке (в пределах шума), готовая mini-frida без дообучения как третий сигнал (прироста нет, если уже есть дообученные модели), бустинг вместо взвешенной суммы целиком (хуже, чем взвешенная сумма + бустинг).

**Open-source.** pandas, numpy, scipy, scikit-learn (CountVectorizer, TfidfVectorizer, HistGradientBoostingClassifier), snowballstemmer, sentence-transformers + PyTorch, модели `sergeyzh/rubert-tiny-turbo` и `sergeyzh/rubert-mini-frida` с Hugging Face.